# Scania APS - Preprocessing
**Arkon Manufacturing AI | Module: ML Classification | Department: Truck Fleet**

**Goal:** Transform raw Scania data into a clean, balanced, model-ready dataset.

Steps:
1. Load data with correct missing value handling
2. Encode target: pos → 1, neg → 0
3. Impute missing values (median strategy)
4. Drop features with >50% missing
5. Apply SMOTE to handle class imbalance
6. Save processed data + scaler

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

plt.style.use('seaborn-v0_8-darkgrid')
ASSETS = 'ml'


## 2. Load Raw Data

In [ ]:
# Load CSV files - 'na' strings are missing values, first 20 rows are description text
df_train = pd.read_csv(RAW_DIR / 'aps_failure_training_set.csv', na_values='na', skiprows=20)
df_test  = pd.read_csv(RAW_DIR / 'aps_failure_test_set.csv',     na_values='na', skiprows=20)

print(f'Train: {df_train.shape} | Test: {df_test.shape}')
print(f'Target values: {df_train["class"].unique()}')

## 3. Encode Target Column

In [ ]:
# Convert string labels to binary: pos (APS failure) = 1, neg (no failure) = 0
df_train['class'] = (df_train['class'] == 'pos').astype(int)
df_test['class']  = (df_test['class']  == 'pos').astype(int)

print(f'Train class distribution after encoding:')
print(df_train['class'].value_counts())

## 4. Separate Features and Target

In [ ]:
# Split dataframes into feature matrix X and target vector y
X_train_raw = df_train.drop(columns=['class'])
y_train     = df_train['class'].values

X_test_raw  = df_test.drop(columns=['class'])
y_test      = df_test['class'].values

print(f'X_train: {X_train_raw.shape} | X_test: {X_test_raw.shape}')

## 5. Drop High-Missing Features

In [ ]:
# Remove features with more than 50% missing values - too unreliable for imputation
missing_rate = X_train_raw.isnull().mean()
cols_to_drop = missing_rate[missing_rate > MISSING_THRESHOLD].index.tolist()

X_train_raw.drop(columns=cols_to_drop, inplace=True)
X_test_raw.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped {len(cols_to_drop)} high-missing features: {cols_to_drop}')
print(f'Remaining features: {X_train_raw.shape[1]}')

## 6. Impute Missing Values

In [ ]:
# Fill remaining missing values with column median - robust to outliers
# Imputer is fit on train only, then applied to test (prevent data leakage)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp  = imputer.transform(X_test_raw)

# Save imputer for use in Streamlit app
joblib.dump(imputer, PROC_DIR / 'imputer_scania.pkl')
print(f'Imputation done. No missing values remaining: {np.isnan(X_train_imp).sum() == 0}')

## 7. Normalise Features

In [ ]:
# Scale all features to [0, 1] range - fit on train only to prevent data leakage
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# Save scaler for Streamlit app inference
joblib.dump(scaler, PROC_DIR / 'scaler_scania.pkl')
print(f'Scaling done. Feature range: [{X_train_scaled.min():.2f}, {X_train_scaled.max():.2f}]')

## 8. Apply SMOTE - Handle Class Imbalance

In [ ]:
# SMOTE generates synthetic minority class samples to balance the dataset
# Applied only to training data - test set stays untouched (real-world distribution)
print(f'Before SMOTE: {pd.Series(y_train).value_counts().to_dict()}')

smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f'After SMOTE:  {pd.Series(y_train_balanced).value_counts().to_dict()}')
print(f'New training size: {X_train_balanced.shape}')

In [ ]:
# Visualise class balance before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

pd.Series(y_train).value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='white')
axes[0].set_title('Before SMOTE')
axes[0].set_xticklabels(['neg (0)', 'pos (1)'], rotation=0)

pd.Series(y_train_balanced).value_counts().plot(
    kind='bar', ax=axes[1], color=['steelblue','tomato'], edgecolor='white')
axes[1].set_title('After SMOTE')
axes[1].set_xticklabels(['neg (0)', 'pos (1)'], rotation=0)

plt.suptitle('Class Balance Before and After SMOTE')
plt.tight_layout()
save_figure(fig, 'scania_preprocessing_plot', subfolder=ASSETS)
plt.show()

## 9. Save Processed Data

In [ ]:
# Get feature names after dropping high-missing columns
feature_names = X_train_raw.columns.tolist()

# Save balanced training set and processed test set as CSV
pd.DataFrame(X_train_balanced, columns=feature_names).assign(
    label=y_train_balanced
).to_csv(PROC_DIR / 'train_scania_processed.csv', index=False)

pd.DataFrame(X_test_scaled, columns=feature_names).assign(
    label=y_test
).to_csv(PROC_DIR / 'test_scania_processed.csv', index=False)

# Save feature names list for model interpretation later
joblib.dump(feature_names, PROC_DIR / 'feature_names_scania.pkl')

print('Saved:')
print(f'  train_scania_processed.csv - {X_train_balanced.shape}')
print(f'  test_scania_processed.csv  - {X_test_scaled.shape}')
print(f'  imputer_scania.pkl')
print(f'  scaler_scania.pkl')
print(f'  feature_names_scania.pkl')

## 10. Summary

| Step | Action | Result |
|------|--------|--------|
| Encode target | pos→1, neg→0 | Binary labels |
| Drop high-missing | >50% threshold | Fewer but cleaner features |
| Impute | Median strategy | No more NaN values |
| Normalise | MinMaxScaler on train | All features in [0, 1] |
| SMOTE | Oversample minority class | Balanced training set |

**Next step:** `03_scania_modeling.ipynb` - train XGBoost, optimise threshold, log to MLflow.